In [2]:
import os
import numpy as np
import pandas as pd

def generate_market_data(num_cities=50, seed=42):
    """
    Generates synthetic global city datasets across macroeconomic and commercial vectors.
    Injects specific business archetypes ('The Trap' and 'The Hidden Gem') to test 
    prescriptive analytics optimization.
    """
    np.random.seed(seed)
    
    # 1. Base Structure Initialization
    region_ids = [f"REG_{i:03d}" for i in range(1, num_cities + 1)]
    
    # Pool of realistic global countries and city names for variance
    countries_cities = [
        ("USA", "Metroplex"), ("DEU", "Industrial Hub"), ("JPN", "Tech Capital"),
        ("BRA", "Coastal Metropolis"), ("GBR", "Financial Core"), ("AUS", "Harbor City"),
        ("IND", "Innovation Center"), ("ZAF", "Commercial Gateway"), ("CAN", "Northern Port"),
        ("FRA", "Cultural Hub")
    ]
    
    countries = []
    cities = []
    for i in range(num_cities):
        cc = countries_cities[i % len(countries_cities)]
        countries.append(cc[0])
        # Append index to keep city names unique
        cities.append(f"{cc[1]} {i+1}")

    # 2. Generate Baseline Distributions using NumPy
    # Economic Baseline
    gdp_growth_rate = np.random.uniform(-0.02, 0.06, num_cities)        # -2% to +6%
    inflation_rate = np.random.uniform(0.01, 0.08, num_cities)         # 1% to 8%
    currency_volatility_index = np.random.uniform(0.1, 0.6, num_cities) # 0.1 to 0.6
    
    # Commercial Baseline
    historical_sales_usd = np.random.uniform(1_000_000, 8_000_000, num_cities)
    competitor_density_score = np.random.randint(3, 9, size=num_cities) # Scale 1 to 10
    customer_acquisition_cost_usd = np.random.uniform(150, 450, num_cities)

    # 3. Inject Strategic Archetypes (Forcing outliers to test pipeline integrity)
    
    # Archetype A: "The Trap" (High historical sales, exploding inflation, crashing GDP)
    # Target: Indices 10 and 35
    for trap_idx in [10, 35]:
        if trap_idx < num_cities:
            cities[trap_idx] = f"Cooling Mega-City (Trap {trap_idx})"
            historical_sales_usd[trap_idx] = np.random.uniform(12_000_000, 15_000_000) # Massive sales
            gdp_growth_rate[trap_idx] = np.random.uniform(-0.05, -0.03)                # Crashing economy
            inflation_rate[trap_idx] = np.random.uniform(0.15, 0.25)                   # Hyperinflation
            currency_volatility_index[trap_idx] = np.random.uniform(0.8, 0.95)         # Extreme risk

    # Archetype B: "The Hidden Gem" (Moderate sales, low competition, stable high GDP growth)
    # Target: Indices 15 and 42
    for gem_idx in [15, 42]:
        if gem_idx < num_cities:
            cities[gem_idx] = f"Emerging Tech Hub (Gem {gem_idx})"
            historical_sales_usd[gem_idx] = np.random.uniform(3_000_000, 4_500_000)   # Moderate footprint
            competitor_density_score[gem_idx] = np.random.randint(1, 3)                # Low competition
            gdp_growth_rate[gem_idx] = np.random.uniform(0.045, 0.055)                 # Stable ~5% growth
            inflation_rate[gem_idx] = np.random.uniform(0.015, 0.03)                   # Low, stable inflation
            customer_acquisition_cost_usd[gem_idx] = np.random.uniform(80, 130)        # Highly efficient CAC

    # 4. Construct DataFrames according to Data Architecture Schema
    macro_df = pd.DataFrame({
        'region_id': region_ids,
        'country': countries,
        'city': cities,
        'gdp_growth_rate': np.round(gdp_growth_rate, 4),
        'inflation_rate': np.round(inflation_rate, 4),
        'currency_volatility_index': np.round(currency_volatility_index, 2)
    })
    
    sales_df = pd.DataFrame({
        'region_id': region_ids,
        'historical_sales_usd': np.round(historical_sales_usd, 2),
        'competitor_density_score': competitor_density_score,
        'customer_acquisition_cost_usd': np.round(customer_acquisition_cost_usd, 2)
    })
    
    return macro_df, sales_df

def save_datasets(macro_df, sales_df, output_dir='data'):
    """Ensures directories exist and exports files to designated CSVs."""
    os.makedirs(output_dir, exist_ok=True)
    
    macro_path = os.path.join(output_dir, 'raw_macro_trends.csv')
    sales_path = os.path.join(output_dir, 'raw_sales_velocity.csv')
    
    macro_df.to_csv(macro_path, index=False)
    sales_df.to_csv(sales_path, index=False)
    
    print(f" Successfully generated and saved raw datasets:")
    print(f"  -> {macro_path} ({len(macro_df)} records)")
    print(f"  -> {sales_path} ({len(sales_df)} records)")

if __name__ == "__main__":
    # Run data factory execution engine
    macro_data, sales_data = generate_market_data(num_cities=50)
    save_datasets(macro_data, sales_data)

 Successfully generated and saved raw datasets:
  -> data\raw_macro_trends.csv (50 records)
  -> data\raw_sales_velocity.csv (50 records)


In [ ]:
import os
import duckdb

def initialize_database(db_path='data/sales_optimizer.db', data_dir='data'):
    """
    Initializes a persistent DuckDB instance, ingests raw CSV components,
    and constructs the relational analytical layer for market optimization.
    """
    print(f"Connecting to database at: {db_path}")
    con = duckdb.connect(db_path)
    
    # Define file target paths
    macro_csv = os.path.join(data_dir, 'raw_macro_trends.csv')
    sales_csv = os.path.join(data_dir, 'raw_sales_velocity.csv')
    
    # 1. Ingest Raw Datasets into Staging Tables
    print("Ingesting staging datasets into relational tables...")
    con.execute(f"""
        CREATE OR REPLACE TABLE staging_macro AS 
        SELECT * FROM read_csv_auto('{macro_csv}');
    """)
    
    con.execute(f"""
        CREATE OR REPLACE TABLE staging_sales AS 
        SELECT * FROM read_csv_auto('{sales_csv}');
    """)
    
    # 2. Build the Analytical Normalization View
    # Normalization formula: (x - min) / (max - min)
    # For cost/density metrics, the inverse framework will be handled in Sprint 3.
    print("Creating normalized analytical view: v_normalized_market_metrics...")
    con.execute("""
        CREATE OR REPLACE VIEW v_normalized_market_metrics AS 
        WITH bounds AS (
            SELECT 
                MIN(m.gdp_growth_rate) AS min_gdp, MAX(m.gdp_growth_rate) AS max_gdp,
                MIN(m.inflation_rate) AS min_inf, MAX(m.inflation_rate) AS max_inf,
                MIN(s.historical_sales_usd) AS min_sales, MAX(s.historical_sales_usd) AS max_sales,
                MIN(s.competitor_density_score) AS min_comp, MAX(s.competitor_density_score) AS max_comp
            FROM staging_macro m
            JOIN staging_sales s ON m.region_id = s.region_id
        )
        SELECT 
            m.region_id,
            m.country,
            m.city,
            s.historical_sales_usd,
            
            -- Min-Max Normalization Calculations
            CASE 
                WHEN (b.max_sales - b.min_sales) = 0 THEN 0.0
                ELSE ROUND((s.historical_sales_usd - b.min_sales) / (b.max_sales - b.min_sales), 4)
            END AS norm_sales,
            
            CASE 
                WHEN (b.max_gdp - b.min_gdp) = 0 THEN 0.0
                ELSE ROUND((m.gdp_growth_rate - b.min_gdp) / (b.max_gdp - b.min_gdp), 4)
            END AS norm_gdp,
            
            CASE 
                WHEN (b.max_inf - b.min_inf) = 0 THEN 0.0
                ELSE ROUND((m.inflation_rate - b.min_inf) / (b.max_inf - b.min_inf), 4)
            END AS norm_inflation,
            
            CASE 
                WHEN (b.max_comp - b.min_comp) = 0 THEN 0.0
                ELSE ROUND((s.competitor_density_score - b.min_comp) / (b.max_comp - b.min_comp), 4)
            END AS norm_competition,
            
            s.customer_acquisition_cost_usd
            
        FROM staging_macro m
        JOIN staging_sales s ON m.region_id = s.region_id
        CROSS JOIN bounds b;
    """)
    
    # Quick structural check
    sample = con.execute("SELECT * FROM v_normalized_market_metrics LIMIT 3").fetchall()
    print(" Database and Analytical Views initialized successfully.")
    con.close()

if __name__ == "__main__":
    initialize_database()

In [7]:
import os
import duckdb
import pandas as pd

def run_market_optimization(db_path='data/sales_optimizer.db'):
    """
    Queries the DuckDB layer, executes the composite scoring algorithm,
    and isolates optimal targets based on strict executive constraints.
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database missing at {db_path}. Please execute db_manager.py first.")
        
    con = duckdb.connect(db_path)
    
    # 1. Extract Normalized Feature Data from DuckDB
    query = "SELECT * FROM v_normalized_market_metrics;"
    df = con.query(query).df()
    con.close()
    
    # 2. Vectorized Math Engine: Composite Market Scoring
    # Inverting inflation and competition scores because lower values are favorable
    df['market_score'] = (
        (0.35 * df['norm_sales']) +
        (0.35 * df['norm_gdp']) +
        (0.15 * (1.0 - df['norm_inflation'])) +
        (0.15 * (1.0 - df['norm_competition']))
    )
    df['market_score'] = df['market_score'].round(4)
    
    # 3. Apply Executive Strategic Hard Constraints
    # Rule 1: CAC must be under $300 USD
    # Rule 2: Inflation must not exceed 10% (0.10)
    df['passed_constraints'] = (df['customer_acquisition_cost_usd'] <= 300.0) & (df['norm_inflation'] < 0.50) # relative scale: upper 50% market rejection
    
    # Sort by score descending
    df = df.sort_values(by='market_score', ascending=False).reset_index(drop=True)
    return df

def generate_executive_report(df, output_path='reports/executive_summary.md'):
    """Compiles analytical outcomes into a clean, scannable Markdown briefing."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Isolate targets
    viable_markets = df[df['passed_constraints']].head(5)
    traps_identified = df[df['city'].str.contains('Trap', case=False)]
    gems_identified = df[df['city'].str.contains('Gem', case=False)]
    
    markdown_content = fr"""# Executive Briefing: Global Market Expansion Optimization
**Data Pipeline Status:** Production Ready  
**Analysis Framework:** Composite Risk/Velocity Optimization (Weights: 35% Sales, 35% GDP, 15% Inf, 15% Comp)

---

## Strategic Recommendation: Top 5 Expansion Targets
These territories cleared all strategic guardrails (CAC $\le$ $300, Inflation Risk < High) and offer the peak efficiency returns.

| Rank | City | Country | Historical Sales | Market Score | CAC |
| :--- | :--- | :--- | :--- | :--- | :--- |
{chr(10).join([f"| {i+1} | {row['city']} | {row['country']} | ${row['historical_sales_usd']:,.2f} | **{row['market_score']:.4f}** | ${row['customer_acquisition_cost_usd']:.2f} |" for i, row in viable_markets.iterrows()])}

---

## Pipeline Stress-Test: Archetype Validations

### 1. The High-Inflation Traps (High Volume / Exploding Macro Risk)
*Goal: The framework must flag and bypass these high-performing vanity revenue markets.*

| City | Historical Sales | Norm Inflation | Passed Constraints? | Market Score |
| :--- | :--- | :--- | :--- | :--- |
{chr(10).join([f"| {row['city']} | ${row['historical_sales_usd']:,.2f} | {row['norm_inflation']:.2f} | {row['passed_constraints']} | {row['market_score']:.4f} |" for _, row in traps_identified.iterrows()])}

### 2. The Hidden Gems (Moderate Volume / Low Competition / Stable Macro)
*Goal: Identify under-the-radar, high-efficiency expansion vectors.*

| City | Historical Sales | Norm Competition | CAC | Market Score |
| :--- | :--- | :--- | :--- | :--- |
{chr(10).join([f"| {row['city']} | ${row['historical_sales_usd']:,.2f} | {row['norm_competition']:.2f} | ${row['customer_acquisition_cost_usd']:.2f} | **{row['market_score']:.4f}** |" for _, row in gems_identified.iterrows()])}

---
*Report auto-generated by the Sales Optimization Engine.*
"""

    with open(output_path, 'w') as f:
        f.write(markdown_content)
    print(f" Executive Markdown Report successfully exported to: {output_path}")

if __name__ == "__main__":
    results_df = run_market_optimization()
    generate_executive_report(results_df)

 Executive Markdown Report successfully exported to: reports/executive_summary.md
